# 05 HumAID: does the classifier hold up on a disaster it has not seen?

Member 5. Notebooks 01 and 02 score the classifiers on the official HumAID test split (TF-IDF baseline macro-F1 0.714, fine-tuned transformer 0.750). That split is built from all 19 disaster events together, so every event that appears in the test split also appears in the training split. The tweets of a new flood will use new place names, new hashtags and new damage details. This notebook measures how much the score drops when a **whole event is held out**.

**The design is paired.** For each event we score the *same test tweets* two ways:

| protocol | training data | test data |
| --- | --- | --- |
| random (the official split) | official train split of all 19 events | official test tweets of event E |
| held out | official train split of the other 18 events | official test tweets of event E |

Only one thing differs: whether the model has seen other tweets from event E. The difference in macro-F1 is the cost of a new disaster.

**Runs on CPU** (about 10 to 25 minutes for all 19 events with the TF-IDF baseline). Part B is an optional GPU run of the transformer on two flood events and is switched off by default (`RUN_TRANSFORMER = False`). Needs internet for the first download only. Clear outputs before committing.

**Data.** `QCRI/HumAID-all`, which notebook 01 used, has no event column. This notebook uses the per-event copy `QCRI/HumAID-events` instead (one configuration per event, with the same train, dev and test files). Licence: CrisisNLP terms of use, same as notebook 01.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "scikit-learn", "pyarrow", "joblib"], check=True)

import os, time
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
DATA = f"{ROOT}/humaid"                       # parquet files saved by notebook 01, model saved by notebook 02
EVENTS_REPO = "QCRI/HumAID-events"
EVENT_NAMES = None                            # None: every configuration of the repository. Or a list to restrict.
SEED = 42
N_JOBS = -1                                   # parallel folds for the TF-IDF part; set 1 if memory runs out

# classes the priority list cares about; keep identical to notebook 02
ACTIONABLE = ["infrastructure_and_utility_damage", "injured_or_dead_people", "displaced_people_and_evacuations",
              "requests_or_urgent_needs", "missing_or_found_people", "caution_and_advice"]

RUN_PART_A = False                             # False: skip the TF-IDF fits and read the table saved by an earlier run
RUN_TRANSFORMER = True                       # Part B: fine-tune the transformer with two flood events held out (GPU)
SMOKE = False                                 # Part B only: tiny run to test the code

## Load every event

In [ ]:
from datasets import load_dataset, get_dataset_config_names

SPLIT_NAMES = {"train": "train", "dev": "validation", "validation": "validation", "val": "validation", "test": "test"}


def event_frame(split, event, split_name):
    """One split of one event as a DataFrame with columns tweet_id, text, label, event, split."""
    df = split.to_pandas()
    text = next(c for c in ("tweet_text", "text", "tweet") if c in df.columns)
    label = next(c for c in ("class_label", "label", "labels") if c in df.columns)
    feat = split.features.get(label)
    if hasattr(feat, "names"):
        df[label] = df[label].map(dict(enumerate(feat.names)))
    return pd.DataFrame({"tweet_id": df["tweet_id"] if "tweet_id" in df.columns else np.nan,
                         "text": df[text], "label": df[label].astype(str), "event": event,
                         "split": SPLIT_NAMES.get(split_name, split_name)})


def load_events(repo=EVENTS_REPO, names=None):
    """Every tweet of every event in one DataFrame (tweet_id, text, label, event, split)."""
    frames = []
    for name in (names or get_dataset_config_names(repo)):
        ds = load_dataset(repo, name, verification_mode="no_checks")
        frames += [event_frame(split, name, split_name) for split_name, split in ds.items()]
    return pd.concat(frames, ignore_index=True)


CACHE = f"{DATA}/events.parquet"
if os.path.exists(CACHE) and EVENT_NAMES is None:
    events = pd.read_parquet(CACHE)
else:
    events = load_events(EVENTS_REPO, EVENT_NAMES)
    if EVENT_NAMES is None:
        events.to_parquet(CACHE, index=False)
print(len(events), "tweets,", events["event"].nunique(), "events")
events.head(3)

## Check the data before using it

Three things matter here. (1) Does every official split contain every event? If so, the official split mixes events and the 0.750 is an in-event score. (2) Does this per-event copy match the copy notebook 01 used? (3) Are there exact duplicate tweets across events or splits? Duplicates between train and test inflate both protocols, so they are removed below.

In [ ]:
def event_table(df):
    """Tweets per event and official split."""
    t = df.pivot_table(index="event", columns="split", values="text", aggfunc="count", fill_value=0)
    t["total"] = t.sum(axis=1)
    return t.sort_values("total", ascending=False)


def split_pooling(df):
    """Number of events that contribute tweets to each official split."""
    return df.groupby("split")["event"].nunique()


def duplicate_texts(df):
    """Rows with an exact duplicate text, and how many of those texts cross event or split boundaries."""
    dup = df[df.duplicated("text", keep=False)]
    return {"rows_with_duplicate_text": len(dup),
            "texts_in_more_than_one_event": int((dup.groupby("text")["event"].nunique() > 1).sum()),
            "texts_in_more_than_one_split": int((dup.groupby("text")["split"].nunique() > 1).sum())}


def agreement_with_notebook01(df, data_dir):
    """Share of the tweets saved by notebook 01 that are found in the same split here. Close to 1 means the two copies match."""
    out = {}
    for split in ("train", "validation", "test"):
        path = f"{data_dir}/{split}.parquet"
        if not os.path.exists(path):
            continue
        ref = pd.read_parquet(path)
        mine = set(df.loc[df["split"] == split, "text"])
        out[split] = {"rows_notebook01": len(ref), "rows_here": int((df["split"] == split).sum()),
                      "share_found": round(float(ref["text"].isin(mine).mean()), 3)}
    return pd.DataFrame(out).T


print(event_table(events))
print("\nevents per split:\n", split_pooling(events).to_string())
print("\nduplicates:", duplicate_texts(events))
print("\nagreement with notebook 01:\n", agreement_with_notebook01(events, DATA))

In [ ]:
SPLIT_RANK = {"train": 0, "validation": 1, "test": 2}


def deduplicate(df):
    """Keep the first occurrence of each exact text, preferring train over validation over test, so no text is in two splits."""
    ordered = df.assign(_rank=df["split"].map(SPLIT_RANK)).sort_values("_rank", kind="stable")
    return ordered.drop_duplicates("text").drop(columns="_rank").reset_index(drop=True)


def hazard_of(event):
    """Hazard type from the event name: flood, earthquake, wildfire, storm or other."""
    name = event.lower()
    for key, hazard in (("flood", "flood"), ("earthquake", "earthquake"), ("quake", "earthquake"),
                        ("wildfire", "wildfire"), ("fire", "wildfire"),
                        ("hurricane", "storm"), ("cyclone", "storm"), ("typhoon", "storm")):
        if key in name:
            return hazard
    return "other"


data = deduplicate(events)
print(len(events), "->", len(data), "tweets after removing exact duplicate texts")
print({e: hazard_of(e) for e in sorted(data["event"].unique())})

## Part A: TF-IDF baseline, all events (CPU)

The model is the notebook 01 baseline: word 1 to 2 grams, class-balanced logistic regression. It is refitted from scratch for each held-out event, including the vocabulary, so nothing from the held-out event reaches the model.

Scores are macro-F1 over the classes that occur in the event's test tweets (a class with no tweets cannot be scored), and macro-F1 over the actionable classes that occur. Small events have few test tweets, so single-event scores are noisy; look at the average over events and at the sign of the drop.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from joblib import Parallel, delayed


def train_tfidf(texts, labels, seed=SEED):
    """TF-IDF (word 1-2 grams) with a class-balanced logistic regression, as in notebook 01."""
    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed))
    return model.fit(texts, labels)


def random_protocol(df):
    """Train once on the official train split of all events; predict the official test split."""
    train, test = df[df["split"] == "train"], df[df["split"] == "test"]
    model = train_tfidf(train["text"], train["label"])
    return test.assign(pred=model.predict(test["text"]))[["event", "label", "pred"]]


def fit_heldout_event(df, event):
    """Train on the official train split of every other event; predict the official test split of the held-out event."""
    train = df[(df["split"] == "train") & (df["event"] != event)]
    test = df[(df["split"] == "test") & (df["event"] == event)]
    model = train_tfidf(train["text"], train["label"])
    return test.assign(pred=model.predict(test["text"]))[["event", "label", "pred"]]


def heldout_protocol(df, events=None, n_jobs=N_JOBS):
    """Predictions for every held-out event, one fit per event."""
    events = list(events) if events is not None else sorted(df["event"].unique())
    parts = Parallel(n_jobs=n_jobs)(delayed(fit_heldout_event)(df, e) for e in events)
    return pd.concat(parts, ignore_index=True)


if RUN_PART_A:
    t0 = time.time()
    pred_random = random_protocol(data)
    pred_heldout = heldout_protocol(data)
    print("done in", round((time.time() - t0) / 60, 1), "minutes;", len(pred_heldout), "held-out predictions")
else:
    pred_random = pred_heldout = None
    print("Part A fits skipped (RUN_PART_A = False): the saved table is read in the next cells.")

In [ ]:
def event_scores(y_true, y_pred):
    """Macro-F1 over the classes in y_true, and over the actionable classes in y_true."""
    present = sorted(set(y_true))
    actionable = [c for c in ACTIONABLE if c in present]
    return {"macro_f1": f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
            "actionable_f1": (f1_score(y_true, y_pred, labels=actionable, average="macro", zero_division=0)
                              if actionable else np.nan)}


def per_event_table(pred_random, pred_heldout):
    """One row per held-out event: the same test tweets scored under both protocols, and the drop."""
    rows = []
    for event, held in pred_heldout.groupby("event"):
        rand = pred_random[pred_random["event"] == event]
        a, b = event_scores(rand["label"], rand["pred"]), event_scores(held["label"], held["pred"])
        rows.append({"event": event, "hazard": hazard_of(event), "n_test": len(held),
                     "macro_f1_random": a["macro_f1"], "macro_f1_heldout": b["macro_f1"],
                     "actionable_f1_random": a["actionable_f1"], "actionable_f1_heldout": b["actionable_f1"]})
    table = pd.DataFrame(rows)
    table["drop_macro"] = table["macro_f1_random"] - table["macro_f1_heldout"]
    table["drop_actionable"] = table["actionable_f1_random"] - table["actionable_f1_heldout"]
    return table.sort_values("drop_macro", ascending=False).reset_index(drop=True)


def gap_summary(table, pred_random, pred_heldout, n_boot=2000, seed=SEED):
    """Pooled macro-F1 under both protocols, mean per-event scores, and the mean drop with a bootstrap interval over events."""
    rng = np.random.default_rng(seed)
    drops = table["drop_macro"].to_numpy()
    boots = [rng.choice(drops, size=len(drops), replace=True).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    same = pred_random[pred_random["event"].isin(table["event"])]
    pooled = lambda p: f1_score(p["label"], p["pred"], average="macro", zero_division=0)
    return {"events": len(table),
            "pooled_macro_f1_random": round(float(pooled(same)), 3),
            "pooled_macro_f1_heldout": round(float(pooled(pred_heldout)), 3),
            "mean_event_macro_f1_random": round(float(table["macro_f1_random"].mean()), 3),
            "mean_event_macro_f1_heldout": round(float(table["macro_f1_heldout"].mean()), 3),
            "mean_drop_macro": round(float(drops.mean()), 3),
            "mean_drop_ci95": (round(float(lo), 3), round(float(hi), 3)),
            "events_with_drop": int((drops > 0).sum()),
            "mean_drop_actionable": round(float(table["drop_actionable"].mean()), 3)}


def drop_by_hazard(table):
    """Mean drop per hazard type (events are the unit, so keep n small in mind)."""
    return (table.groupby("hazard").agg(events=("event", "count"), mean_drop_macro=("drop_macro", "mean"),
                                        min_drop=("drop_macro", "min"), max_drop=("drop_macro", "max"),
                                        mean_drop_actionable=("drop_actionable", "mean")).round(3))


if RUN_PART_A:
    table = per_event_table(pred_random, pred_heldout)
    summary = gap_summary(table, pred_random, pred_heldout)
else:
    table = pd.read_csv(f"{DATA}/event_holdout_tfidf.csv")
    summary = {"note": "Part A not rerun; per-event table read from event_holdout_tfidf.csv"}
print(summary)
print(drop_by_hazard(table))
table.round(3)

In [ ]:
import matplotlib.pyplot as plt


def plot_event_gap(table, title):
    """One row per event: the held-out macro-F1 and the random-split macro-F1 joined by a line."""
    t = table.sort_values("drop_macro")
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(t) + 1.5))
    y = np.arange(len(t))
    ax.hlines(y, t["macro_f1_heldout"], t["macro_f1_random"], color="0.7")
    ax.scatter(t["macro_f1_random"], y, label="random split", zorder=3)
    ax.scatter(t["macro_f1_heldout"], y, label="event held out", zorder=3)
    ax.set_yticks(y, t["event"])
    ax.set_xlabel("macro-F1 on the same test tweets")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


plot_event_gap(table, "TF-IDF baseline: random split vs event held out");

In [ ]:
if RUN_PART_A:
    table.to_csv(f"{DATA}/event_holdout_tfidf.csv", index=False)
    print("saved", f"{DATA}/event_holdout_tfidf.csv")

## Part B (optional, GPU): the fine-tuned transformer with two flood events held out

Part A tells us how much a new event costs the baseline. The 0.750 comes from the transformer, so check it directly, but on two events only because each fit is a full fine-tune. The two largest flood events are used because the project is about floods.

**What is compared.** For each held-out flood event, (1) the saved notebook 02 model scored on that event's test tweets (the random-split protocol; it has seen other tweets of that event), and (2) a new model fine-tuned on the training tweets of every other event (the best epoch is chosen on the validation tweets of the other events only). The two models share the recipe of notebook 02: same checkpoint, learning rate, class weights and seed. Estimated time: roughly 10 to 20 minutes per held-out event on a T4 (an estimate, not measured). Run with `SMOKE = True` first. Set `RUN_TRANSFORMER = True` in the settings cell to run it.

In [ ]:
from sklearn.metrics import f1_score

MODEL_NAME = "cardiffnlp/twitter-roberta-base"
MAX_LEN, BATCH, EPOCHS, LR, WEIGHT_POWER = 128, 32, 3, 2e-5, 1.0
HELD_OUT = None                  # None: the two largest flood events. Or a list of event names.


def flood_events(df, n=2):
    """The n flood events with the most tweets."""
    flood = [e for e in df["event"].unique() if hazard_of(e) == "flood"]
    return sorted(flood, key=lambda e: -int((df["event"] == e).sum()))[:n]


def encode_examples(tokenizer, df, label2id, max_len):
    """One dict per tweet with input_ids, attention_mask and labels."""
    enc = tokenizer(list(df["text"]), truncation=True, max_length=max_len)
    y = df["label"].map(label2id).tolist()
    return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": y[i]}
            for i in range(len(df))]


def make_loader(tokenizer, examples, batch, shuffle):
    """DataLoader that pads each batch to its longest tweet."""
    import torch

    def collate(items):
        enc = tokenizer.pad([{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in items],
                            return_tensors="pt")
        enc["labels"] = torch.tensor([x["labels"] for x in items])
        return enc
    return torch.utils.data.DataLoader(examples, batch_size=batch, shuffle=shuffle, collate_fn=collate)


def predict_ids(model, loader, device):
    """Predicted class ids and true ids for one loader."""
    import torch
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for b in loader:
            y = b.pop("labels")
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(**b).logits
            preds.append(logits.float().argmax(-1).cpu())
            ys.append(y)
    return torch.cat(preds).numpy(), torch.cat(ys).numpy()


def fit_transformer(train_df, val_df, labels, epochs, device, seed=SEED):
    """Fine-tune on train_df, keep the epoch with the best validation macro-F1, return model and tokenizer."""
    import random, torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    label2id = {l: i for i, l in enumerate(labels)}
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_loader = make_loader(tokenizer, encode_examples(tokenizer, train_df, label2id, MAX_LEN), BATCH, True)
    val_loader = make_loader(tokenizer, encode_examples(tokenizer, val_df, label2id, MAX_LEN), BATCH * 2, False)
    counts = np.bincount(train_df["label"].map(label2id).to_numpy(), minlength=len(labels)).clip(min=1)
    weights = torch.tensor((len(train_df) / (len(labels) * counts)) ** WEIGHT_POWER, dtype=torch.float32).to(device)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    best, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for b in train_loader:
            y = b.pop("labels").to(device)
            b = {k: v.to(device) for k, v in b.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(**b).logits
            loss = loss_fn(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
        pred, y_val = predict_ids(model, val_loader, device)
        score = f1_score(y_val, pred, average="macro")
        print("epoch", epoch, "validation macro-F1", round(float(score), 3))
        if score > best:
            best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, tokenizer


def heldout_transformer(df, event, labels, epochs, device, smoke=False):
    """Fine-tune without the held-out event and predict that event's official test tweets."""
    train = df[(df["split"] == "train") & (df["event"] != event)]
    val = df[(df["split"] == "validation") & (df["event"] != event)]
    test = df[(df["split"] == "test") & (df["event"] == event)]
    if smoke:
        train, val = train.sample(n=min(2000, len(train)), random_state=SEED), val.sample(n=min(500, len(val)), random_state=SEED)
    model, tokenizer = fit_transformer(train, val, labels, epochs, device)
    loader = make_loader(tokenizer, encode_examples(tokenizer, test, {l: i for i, l in enumerate(labels)}, MAX_LEN), BATCH * 2, False)
    pred, _ = predict_ids(model, loader, device)
    return test.assign(pred=[labels[i] for i in pred])[["event", "label", "pred"]]


def saved_model_scores(pred_path, df, events):
    """Event for each test prediction saved by notebook 02 (matched on exact text), kept for the chosen events."""
    saved = pd.read_parquet(pred_path)
    lookup = df.drop_duplicates("text").set_index("text")["event"]
    saved = saved.assign(event=saved["text"].map(lookup))
    print("saved predictions matched to an event:", int(saved["event"].notna().sum()), "of", len(saved))
    return saved[saved["event"].isin(events)][["event", "label", "pred"]]

In [ ]:
if RUN_TRANSFORMER:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    assert device == "cuda" or SMOKE, "Part B needs a GPU runtime (Runtime, Change runtime type, T4)."
    LABELS = sorted(data["label"].unique())
    held_out = HELD_OUT or flood_events(data, 2)
    print("held out:", held_out, "| device:", device, "| smoke:", SMOKE)
    parts, t0 = [], time.time()
    for event in held_out:
        parts.append(heldout_transformer(data, event, LABELS, 1 if SMOKE else EPOCHS, device, SMOKE))
        print(event, "done,", round((time.time() - t0) / 60, 1), "minutes so far")
    pred_heldout_tf = pd.concat(parts, ignore_index=True)
    saved_path = f"{DATA}/transformer/test_predictions.parquet"
    pred_random_tf = saved_model_scores(saved_path, events, held_out)
    table_tf = per_event_table(pred_random_tf, pred_heldout_tf)
    if SMOKE:
        print("Smoke run: these numbers come from a tiny subset and are not results.")
    else:
        table_tf.to_csv(f"{DATA}/event_holdout_transformer.csv", index=False)
    print(table_tf.round(3).to_string(index=False))
else:
    print("Part B is switched off (RUN_TRANSFORMER = False).")

In [ ]:
table_tf = per_event_table(pred_random_tf, pred_heldout_tf)
table_tf.to_csv(f"{DATA}/event_holdout_transformer.csv", index=False)
print(table_tf.round(3).to_string(index=False))

## Paste back

Run this cell and paste its output, so the numbers can go into the report and the demo.

In [ ]:
def paste_back(events, data, summary, table, hazards, table_tf=None):
    """One text block with the data checks, the Part A summary and, when it was run, Part B."""
    lines = [f"events: {events['event'].nunique()}; tweets {len(events)} -> {len(data)} after removing duplicate texts",
             "events per split: " + ", ".join(f"{k}={v}" for k, v in split_pooling(events).items()),
             "duplicates: " + str(duplicate_texts(events)),
             "PART A (TF-IDF): " + str(summary), "by hazard:", hazards.to_string(),
             "per event:", table.round(3).to_string(index=False)]
    if table_tf is not None:
        lines += ["PART B (transformer):", table_tf.round(3).to_string(index=False)]
    return "\n".join(lines)


print(paste_back(events, data, summary, table, drop_by_hazard(table),
                 table_tf if RUN_TRANSFORMER and not SMOKE else None))

## Notes for the report
- State the protocol in one sentence: the official HumAID split pools all 19 events, so its scores are in-event scores. Report the held-out score next to it, on the same test tweets, with the number of events behind each average.
- Say what the drop is and is not. It measures the cost of an unseen event for this recipe (TF-IDF and, for two flood events, the transformer). It is not a measure of how the system would do on tweets from a flood in a new country or in another language.
- The unit of analysis is the event. With 19 events (or 2 for the transformer) intervals are wide; give the interval and the number of events where the score fell.
- Report the actionable-class drop as well, because the priority list uses those classes. `missing_or_found_people` is rare, so its per-event scores are noisy.
- The held-out score uses exact-duplicate removal, so the random-protocol baseline here can differ from notebook 01's 0.714 by a small amount. Quote the paired numbers from this notebook, not a mix of the two.
- Do not tune anything on the held-out events. They were used for scoring only.